In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = pd.read_csv(
    "../data/cleaned/telco_customer_churn_cleaned.csv"
)

# Remove ID
data = data.drop(columns=["customerID"])

# Separate features and target
X = data.drop(columns=["Churn"])
y = data["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Identify feature types
numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    include="str"
).columns.tolist()

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [2]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            StandardScaler(),
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

In [3]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [5]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter= 1000,
        random_state= 42
    ),

    "Decision tree": DecisionTreeClassifier(
        random_state= 42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators= 200,
        random_state= 42,
        n_jobs= -1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state= 42
    )
}

In [6]:
for name, model in models.items():
    model.fit(
        X_train_processed,
        y_train
    )
    print(f"\n {name} model has trained Succefssfully ")


 Logistic Regression model has trained Succefssfully 

 Decision tree model has trained Succefssfully 

 Random Forest model has trained Succefssfully 

 Gradient Boosting model has trained Succefssfully 


In [7]:
model_predictions = {}

for name, model in models.items():
    
    y_pred = model.predict(X_test_processed)
    y_probability = model.predict_proba(X_test_processed)[:, 1]
    
    model_predictions[name] = {
        "y_pred": y_pred,
        "y_probability": y_probability
    }
    
    print(f"{name} predictions generated successfully")

Logistic Regression predictions generated successfully
Decision tree predictions generated successfully
Random Forest predictions generated successfully
Gradient Boosting predictions generated successfully


In [8]:
model_result = {}

for name in models:

    y_pred = model_predictions[name]["y_pred"]
    y_probability = model_predictions[name]["y_probability"]

    results = pd.DataFrame(
        {
            "Actual": y_test.values,
            "predicted": y_pred,
            "Churn_Probability": y_probability
        }
    )

    model_result[name]= results

    print(f"\n\n {name} ")
    print(results.head())



 Logistic Regression 
   Actual  predicted  Churn_Probability
0       0          0           0.045920
1       0          1           0.683713
2       0          0           0.058940
3       0          0           0.401835
4       0          0           0.021424


 Decision tree 
   Actual  predicted  Churn_Probability
0       0          0                0.0
1       0          1                1.0
2       0          0                0.0
3       0          1                1.0
4       0          0                0.0


 Random Forest 
   Actual  predicted  Churn_Probability
0       0          0              0.000
1       0          1              0.725
2       0          0              0.070
3       0          0              0.305
4       0          0              0.015


 Gradient Boosting 
   Actual  predicted  Churn_Probability
0       0          0           0.023600
1       0          1           0.836548
2       0          0           0.065688
3       0          0           0.32234

In [9]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [10]:
evaluation_results = []

for name, model in models.items():

    # predictions
    y_pred = model.predict(X_test_processed)

    # churn Probablities
    y_probablities = model.predict_proba(X_test_processed)[:, 1]

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_probability)

    # confusion Matrics
    cm = confusion_matrix(y_test, y_pred)

    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nAccuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)
    print("ROC-AUC  :", roc_auc)

    evaluation_results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })


Logistic Regression

Confusion Matrix:
[[926 109]
 [165 209]]

Accuracy : 0.8055358410220014
Precision: 0.6572327044025157
Recall   : 0.5588235294117647
F1 Score : 0.6040462427745664
ROC-AUC  : 0.8432754656539823

Decision tree

Confusion Matrix:
[[832 203]
 [190 184]]

Accuracy : 0.7210787792760823
Precision: 0.4754521963824289
Recall   : 0.4919786096256685
F1 Score : 0.4835742444152431
ROC-AUC  : 0.8432754656539823

Random Forest

Confusion Matrix:
[[924 111]
 [194 180]]

Accuracy : 0.7835344215755855
Precision: 0.6185567010309279
Recall   : 0.48128342245989303
F1 Score : 0.5413533834586466
ROC-AUC  : 0.8432754656539823

Gradient Boosting

Confusion Matrix:
[[938  97]
 [181 193]]

Accuracy : 0.8026969481902059
Precision: 0.6655172413793103
Recall   : 0.516042780748663
F1 Score : 0.5813253012048193
ROC-AUC  : 0.8432754656539823


In [13]:
comparison_results = []

for name, model in models.items():

    y_pred = model.predict(X_test_processed)
    y_probability = model.predict_proba(X_test_processed)[:, 1]

    comparison_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_probability)
    })

comparison_df = pd.DataFrame(comparison_results)

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.805536,0.657233,0.558824,0.604046,0.842135
1,Decision tree,0.721079,0.475452,0.491979,0.483574,0.647676
2,Random Forest,0.783534,0.618557,0.481283,0.541353,0.820602
3,Gradient Boosting,0.802697,0.665517,0.516043,0.581325,0.843275


In [14]:
comparison_df.sort_values(
    by="ROC-AUC",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
3,Gradient Boosting,0.802697,0.665517,0.516043,0.581325,0.843275
0,Logistic Regression,0.805536,0.657233,0.558824,0.604046,0.842135
2,Random Forest,0.783534,0.618557,0.481283,0.541353,0.820602
1,Decision tree,0.721079,0.475452,0.491979,0.483574,0.647676
